# Assignment Solution: Gaussian Process Regression and Linear Regression

This notebook contains a complete solution structure, code, analysis, and discussion for the assignment.

## Part 1 - Gaussian Process Regression (Energy Efficiency Dataset)
Goal: Predict Heating Load (Y1) and Cooling Load (Y2) using Gaussian Process Regression.

In [ ]:

import pandas as pd
import numpy as np
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, r2_score

path = kagglehub.dataset_download("elikplim/eergy-efficiency-dataset")
df = pd.read_csv(path + "/ENB2012_data.csv")

df.head()


In [ ]:

X = df.iloc[:,0:8]
Y = df.iloc[:,8:10]   # Heating Load and Cooling Load

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

kernel = ConstantKernel(1.0) * RBF(length_scale=1.0) + WhiteKernel()

gpr = MultiOutputRegressor(
    GaussianProcessRegressor(kernel=kernel, random_state=42)
)

gpr.fit(X_train, Y_train)

Y_pred = gpr.predict(X_test)

print("Heating Load R²:", r2_score(Y_test.iloc[:,0], Y_pred[:,0]))
print("Cooling Load R²:", r2_score(Y_test.iloc[:,1], Y_pred[:,1]))
print("Overall RMSE:", np.sqrt(mean_squared_error(Y_test, Y_pred)))


### Discussion

- Gaussian Process Regression is suitable because the relationship between building characteristics and energy loads is nonlinear.
- GPR provides both predictions and uncertainty estimates.
- High R² values indicate that the model captures the relationship effectively.
- Heating load and cooling load can be modeled jointly using a multi-output framework.
- Therefore, GPR is a strong candidate for this dataset.


## Part 2 - Linear Regression (Green Building Dataset)
Goal: Predict `predicted_energy_demand` using a suitable linear model.

In [ ]:

import pandas as pd
import numpy as np
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

path = kagglehub.dataset_download(
    "programmer3/green-building-multi-source-environment-dataset"
)

df = pd.read_csv(path + "/green_building_dataset.csv")

df.head()


In [ ]:

target = "predicted_energy_demand"

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

if target in numeric_cols:
    numeric_cols.remove(target)

X = df[numeric_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print("R² Score =", r2_score(y_test, y_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_test, y_pred)))

coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr.coef_
}).sort_values("Coefficient", key=abs, ascending=False)

coefficients.head(15)


### Justification of Feature Selection

- Linear regression requires explanatory variables that have a measurable linear influence on energy demand.
- Numerical environmental, occupancy, and building-performance parameters are therefore selected.
- Features with larger coefficient magnitudes contribute more strongly to the prediction.

### Discussion

- R² indicates how much of the variation in energy demand is explained by the model.
- RMSE measures prediction error in the original units.
- If R² is reasonably high and RMSE is low, a linear model is adequate.
- If performance is poor, nonlinear methods such as Random Forests, Gradient Boosting, or Gaussian Processes may perform better.


# Final Conclusions

### Gaussian Process Regression
- Successfully models nonlinear relationships between building design parameters and energy loads.
- Provides accurate predictions and uncertainty information.
- Appropriate for predicting heating and cooling loads.

### Linear Regression
- Serves as an interpretable baseline model.
- Allows identification of influential parameters affecting energy demand.
- Performance depends on how linear the underlying relationships are.

Overall, Gaussian Process Regression is expected to outperform Linear Regression when nonlinear interactions dominate, while Linear Regression offers simplicity and interpretability.
